# Import packages and list of compounds

In [ ]:
import time
import pandas as pd
from target_extractor.perplex_sdk_pro import *

In [2]:
# either import compounds from a CSV file or use a predefined list
compounds = None # if None, will import from CSV

In [ ]:
if compounds is None:
    # Import a column from an Excel file and convert to list
    excel_path = (
        "../data/compounds_names.xlsx"
    )

    current_time = time.strftime("%Y%m%d-%H%M%S")
    export_dir = (
        "../results"
        + f"/{current_time}_result_pro_test.xlsx"
    )

    column_name = "Compounds"
    df_compounds = pd.read_excel(excel_path)
    COMPOUNDS_LIST = df_compounds[column_name].dropna().tolist()
    # clean the name of the compounds
    COMPOUNDS_LIST = [x.strip(' *"') for x in COMPOUNDS_LIST]
else:
    COMPOUNDS_LIST = compounds  # the names should be cleaned already.

In [ ]:
# Remove duplicated names from the list of compounds, preserving order
COMPOUNDS_LIST = list(dict.fromkeys(COMPOUNDS_LIST)) 

print(f"number of compounds after removing duplicates:")
print(len(COMPOUNDS_LIST))

# Automate the API using Perplexity SDK 


In [ ]:
# how many compounds per prompt
CHUNK_SIZE = 5
all_tables = []
_groups = list(chunks(COMPOUNDS_LIST, CHUNK_SIZE))

for gi, group in enumerate(_groups):
    is_last = (gi == len(_groups) - 1)

    # Modify the prompt as you wish
    prompt = (
        "Make a table for the following compounds and keep the order of them as it is, and name the"
        " column 'Compounds'. "
        "For each compound, make the second column called 'specific_targets' and check whether there is any evidence that they directly "
        "target any of YAP1 (Yes1 associated transcriptional regulator; yes-associated protein 1), or "
        "TAZ (WW domain containing transcription regulator 1; transcriptional coactivator with PDZ-binding motif, also called WWTR1), "
        "or TEAD (TEA domain transcription factors; TEA/ATTS domain family). If any of these three targets are mentioned, "
        "add 'YAP/TAZ-TEAD' in this second column. If there is any evidence for targeting Na+/K+-ATPase "
        "(sodium/potassium adenosine triphosphatase; sodium/potassium pump), add 'Sodium-potas-ATPase' to the column. "
        "If evidence for both of them add both labels separated by ';'. If there is no evidence, add 'None'. "
        "Make the third column called 'extra_explanation' and if the compound has any of the mentioned targets, explain shortly how "
        "the compound targets those molecules. If there is no target identified, add 'None'. "
        "Put the references link (and not number of the reference) for each compound in the fourth column called 'extra_ref' and do not add the"
        "number of the references to the third column. separate multiple references with ';'. "
        "The references can only remain empty if there is no target identified for the compound and you added 'None' to the third column."
        f"{', '.join(group)}"
    )

    messages = [{"role": "user", "content": prompt}]

    # Try/retry the same chunk if the parsed DataFrame has fewer than
    # CHUNK_SIZE rows, except if this is the last loop (which may
    # legitimately have fewer rows).
    max_requeries_for_short_chunk = 5
    attempt = 0
    accepted = False

    while True:
        attempt += 1

        stream = create_stream_with_backoff(
            messages=messages,
            # or "sonar" OR "sonar-deep-research"
            model="sonar-pro",
            # Pro Search behavior
            web_search_options=None,
            # options: academic, sec, web
            search_mode="academic",
        )

        content, references = stream_to_text_and_refs(stream)
        print(f"The content of this search is :\n{content}\n")
        print(f"The references of this search is :\n{references}\n")

        md_table = extract_markdown_table(content)
        if not md_table:
            rows = 0
            df = None
        else:
            df = markdown_to_df(md_table)
            rows = len(df)

        if is_last:
            if not md_table:
                print(f"No table returned for final chunk {group}")
            else:
                all_tables.append(df)
                print(
                    f"Processed {len(all_tables)} chunk(s)"
                    f" of size {CHUNK_SIZE} so far."
                )
            accepted = True
            break
        else:
            if rows < CHUNK_SIZE:
                if attempt <= (1 + max_requeries_for_short_chunk):
                    print(
                        f"Short chunk ({rows} < {CHUNK_SIZE})"
                        f" for {group}. Re-querying attempt"
                        f" {attempt}/{1 + max_requeries_for_short_chunk}..."
                    )
                    time.sleep(1.0)
                    continue
                else:
                    if df is not None:
                        all_tables.append(df)
                        print(
                            f"Accepted short chunk after retries:"
                            f" {rows} rows for {group}."
                        )
                    else:
                        print(
                            f"No table for chunk {group}"
                            " after retries; skipping."
                        )
                    accepted = True
                    break
            else:
                all_tables.append(df)
                print(
                    f"Processed {len(all_tables)} chunk(s)"
                    f" of size {CHUNK_SIZE} so far."
                )
                print(f"----------------------------------")
                accepted = True
                break

    if not accepted:
        print(
            f"Chunk {group} not accepted due to unexpected"
            " conditions; continuing."
        )


# ---------- ANALYSIS / EXPORT ----------
if all_tables:
    # standard column names we want to have
    standard_columns = [
        "Compound",
        "specific_targets",
        "extra_explanation",
        "extra_ref",
    ]
    for i, df in enumerate(all_tables):
        # rename columns to standard names
        df.columns = standard_columns[:len(df.columns)]
        all_tables[i] = df

    combined = pd.concat(all_tables, ignore_index=True)
    combined.to_excel(export_dir, index=False)
else:
    print("No tables produced.")